<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/07_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Engineering for Machine Learning

> **Track:** Data Science / ML Engineer | **Level:** Intermediate | **Time:** 2-3 hours

## Overview
"Feature engineering is the process of using domain knowledge to extract features from raw data." — It's often the single highest-impact activity in an ML project, more so than model selection.

### What You'll Learn
- Handling missing values (multiple strategies)
- Encoding categorical variables
- Feature scaling and normalization
- Feature interactions and polynomial features
- Date/time feature extraction
- Feature selection (mutual info, correlation)
- Building sklearn Pipelines

```bash
pip install pandas numpy scikit-learn matplotlib seaborn
```

In [ ]:
# Setup: create a messy real-world-style dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif

np.random.seed(42)
n = 500

raw_df = pd.DataFrame({
    "age": np.random.randint(18, 70, n).astype(float),
    "income": np.random.exponential(50000, n),
    "job_title": np.random.choice(["Engineer", "Manager", "Analyst", "Director", None], n, p=[0.3,0.25,0.25,0.15,0.05]),
    "department": np.random.choice(["Tech", "Sales", "HR", "Finance"], n),
    "years_employed": np.random.randint(0, 20, n).astype(float),
    "performance_score": np.random.choice([1,2,3,4,5], n),
    "signup_date": pd.date_range("2020-01-01", periods=n, freq="12h"),
    "churn": np.random.choice([0, 1], n, p=[0.75, 0.25])
})

# Introduce missing values
raw_df.loc[np.random.choice(n, 30, replace=False), "age"] = np.nan
raw_df.loc[np.random.choice(n, 50, replace=False), "income"] = np.nan

print(f"Dataset shape: {raw_df.shape}")
print(f"Missing values:\n{raw_df.isnull().sum()[raw_df.isnull().sum() > 0]}")

## 1. Handling Missing Values

In [ ]:
# Multiple imputation strategies and when to use each
df = raw_df.copy()

# Strategy 1: Mean imputation (numeric, MCAR assumption)
df["age_mean_imputed"] = df["age"].fillna(df["age"].mean())

# Strategy 2: Median imputation (robust to outliers)
df["income_median_imputed"] = df["income"].fillna(df["income"].median())

# Strategy 3: Mode imputation for categoricals
df["job_title"] = df["job_title"].fillna(df["job_title"].mode()[0])

# Strategy 4: Add a "was_missing" indicator flag (preserves missingness signal)
df["age_was_missing"] = raw_df["age"].isnull().astype(int)
df["income_was_missing"] = raw_df["income"].isnull().astype(int)

# Strategy 5: KNN imputation (better for correlated features)
from sklearn.impute import KNNImputer
knn_imp = KNNImputer(n_neighbors=5)
df[["age", "income"]] = knn_imp.fit_transform(df[["age", "income"]])

print("Missing values after imputation:")
print(df[["age", "income"]].isnull().sum())
print(f"\nKNN-imputed age mean: {df['age'].mean():.1f} (original mean: {raw_df['age'].mean():.1f})")

## 2. Categorical Encoding

In [ ]:
# Three categorical encoding strategies

# 1. One-Hot Encoding (for low-cardinality, no ordinal relationship)
dept_ohe = pd.get_dummies(df["department"], prefix="dept", dtype=int)
print("1. One-Hot Encoding (department):")
print(dept_ohe.head(3))

# 2. Label Encoding (for ordinal variables)
perf_order = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
df["performance_ordinal"] = df["performance_score"].map(perf_order)
print("\n2. Ordinal Encoding (performance_score → 0-4):")
print(df[["performance_score", "performance_ordinal"]].value_counts().head())

# 3. Target Encoding (for high-cardinality — encode with target mean)
target_mean = df.groupby("job_title")["churn"].mean()
df["job_title_target_enc"] = df["job_title"].map(target_mean)
print("\n3. Target Encoding (job_title → mean churn rate):")
print(target_mean.round(3))

## 3. Feature Creation & Date Features

In [ ]:
# Feature interactions and date-time feature extraction

# Interaction features
df["income_per_year"] = df["income"] / (df["years_employed"] + 1)  # avoid div by zero
df["age_income_ratio"] = df["age"] / (df["income"] / 10000 + 1)
df["seniority_score"] = df["years_employed"] * df["performance_ordinal"]

# Date/time features
df["signup_year"] = df["signup_date"].dt.year
df["signup_month"] = df["signup_date"].dt.month
df["signup_dayofweek"] = df["signup_date"].dt.dayofweek
df["signup_quarter"] = df["signup_date"].dt.quarter
df["days_since_signup"] = (pd.Timestamp.now() - df["signup_date"]).dt.days
df["is_weekend_signup"] = (df["signup_dayofweek"] >= 5).astype(int)

print("New features created:")
new_features = ["income_per_year", "age_income_ratio", "seniority_score",
                "signup_month", "signup_dayofweek", "days_since_signup", "is_weekend_signup"]
print(df[new_features].describe().round(2))

## 4. Feature Selection & Sklearn Pipeline

In [ ]:
# Feature selection with mutual information + build a production Pipeline

feature_cols = ["age", "income", "years_employed", "performance_ordinal",
                "income_per_year", "seniority_score", "days_since_signup",
                "signup_month", "is_weekend_signup", "job_title_target_enc"]

X = df[feature_cols].fillna(0)
y = df["churn"]

# Mutual information scores
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False)

print("Feature Importance (Mutual Information with target):")
print(mi_series.round(4))

# Build a production sklearn Pipeline
numeric_features = ["age", "income", "years_employed", "income_per_year", "days_since_signup"]
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
], remainder="passthrough")

full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=50, random_state=42)),
])

full_pipeline.fit(X, y)
train_score = full_pipeline.score(X, y)
print(f"\nPipeline training accuracy: {train_score:.3f}")
print("Pipeline saves feature engineering steps for deployment — no skew!")

## Exercises

1. **Polynomial features**: Use `sklearn.preprocessing.PolynomialFeatures(degree=2, interaction_only=True)` on the top 4 features by mutual information. Train a logistic regression before and after adding polynomial features. Does accuracy improve? How does the number of features change?

2. **Frequency encoding**: Implement frequency encoding for `job_title` (replace each category with its frequency in the training set). Compare it to target encoding using 5-fold cross-validation AUC-ROC. Which leaks less information on the test set?

3. **Automated feature selection**: Use `sklearn.feature_selection.SelectFromModel` with a Random Forest to automatically select features above median importance. Compare a model trained on all features vs. the selected subset using cross-validated AUC-ROC. Is the pruned model competitive?